# Monday: LLM labels, words and embeddings

We will measure r/AmItheAsshole posts three ways: an LLM label, TF-IDF and embeddings.

Work with a partner. Each task has an example. In the empty cell below it, write a similar line. The check comment tells you what to expect. Make a copy named `week10_live_workspace_mine.ipynb`. The LLM labels are in a file, so you don't need a Gemini key.

In [2]:
import warnings
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore", category=FutureWarning)  # a harmless warning from one of the libraries
pd.set_option("display.max_colwidth", 300)  # show more of each text

## 1. Load the posts

Each post has the community's verdict in its flair: NTA (not the asshole) or YTA (you're the asshole).

In [ ]:
posts = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_posts.csv")
posts[["title", "verdict"]].head()  # Check: 785 rows in posts

In [ ]:
# Example: how many posts have each verdict
posts["verdict"].value_counts()  # NTA 682, YTA 103

In [ ]:
# Your turn: the same as shares. Add normalize=True inside the brackets.
# Always guessing the most common verdict is the baseline. Check: NTA 0.87, YTA 0.13

## 2. An LLM label: does the poster admit fault?

No column says whether a poster admits fault. So 30 posts (15 NTA, 15 YTA) were read and labeled by hand, with the verdicts hidden, using this definition:

- yes: the poster says that something they did was wrong, unfair, or an overreaction, even in passing.
- no: the poster does not admit any fault.

The file has those hand labels (`hand`) and two other labels for the same posts:

- `keyword`: yes if the post contains a phrase like "my fault", "I overreacted" or "I shouldn't have"
- `gemini`: Gemini, with the same definition as its instruction

In [ ]:
labels = pd.read_csv("https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data/week10_post_labels.csv")
labels[["title", "hand", "keyword", "gemini"]].head()  # Check: 30 rows in labels

In [ ]:
# Example: the accuracy of keyword, the share of posts where it agrees with the hand label
(labels["keyword"] == labels["hand"]).mean()  # 0.73

In [ ]:
# Your turn: the accuracy of gemini. Check: 0.83
# Then the accuracy of always guessing "no": the share of posts labeled no in hand. Check: 0.87

### Two kinds of mistakes

Say you are looking for posts where the poster admits fault, so yes is the positive label.

- A **false positive** is a post labeled yes whose hand label is no.
- A **false negative** is a post labeled no whose hand label is yes.

Accuracy counts both kinds the same way.

In [ ]:
# Hand labels in the rows, Gemini's labels in the columns.
# The false positives are in the no row, yes column; the false negatives are in the yes row, no column.
pd.crosstab(labels["hand"], labels["gemini"])

In [ ]:
# Example: false positives of gemini (labeled yes, but the hand label is no)
((labels["gemini"] == "yes") & (labels["hand"] == "no")).sum()  # 4

In [ ]:
# Your turn: false negatives of gemini (labeled no, but the hand label is yes). Check: 1

In [ ]:
# Your turn: false positives and false negatives of keyword. Check: 5 and 3

In [ ]:
# The share of posts each method labels yes, next to the share in the hand labels
(labels[["hand", "keyword", "gemini"]] == "yes").mean()

In [ ]:
# The posts where Gemini and the hand label disagree, with Gemini's reason
labels.loc[labels["gemini"] != labels["hand"], ["title", "hand", "gemini", "gemini_reason"]]

Read two posts where Gemini and the hand label disagree (the full text is in `labels["text"]`). Who is right: the hand label, Gemini, or is the definition unclear?

Your answer:

## 3. TF-IDF: which words go with each verdict?

TF-IDF gives each post one weight per word. The average weights for YTA and for NTA posts show which words go with each verdict.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", min_df=30)  # words in at least 30 posts
weights = pd.DataFrame(tfidf.fit_transform(posts["text"]).toarray(), columns=tfidf.get_feature_names_out())
# one row per word: its average weight in the NTA posts and in the YTA posts
by_verdict = pd.DataFrame({
    "NTA": weights[posts["verdict"] == "NTA"].mean(),
    "YTA": weights[posts["verdict"] == "YTA"].mean(),
})
by_verdict["ratio"] = by_verdict["YTA"] / by_verdict["NTA"]
by_verdict.shape  # Check: (350, 3)

In [ ]:
# Example: the 10 words with the highest ratio, the most typical of YTA posts
by_verdict.sort_values("ratio", ascending=False).head(10)  # children, looks, born, wife, feels, ...

In [ ]:
# Your turn: the 10 words with the lowest ratio. Change one thing.
# Check: bedroom, sick, food, cut, sorry, ...

In [ ]:
# The 6 words with the lowest ratio (NTA) and the 6 with the highest (YTA), as bars
by_ratio = by_verdict.sort_values("ratio")
selected = pd.concat([by_ratio.head(6), by_ratio.tail(6)])
selected[["NTA", "YTA"]].plot.barh(figsize=(6, 5), xlabel="average TF-IDF weight", ylabel="")  # Check: wife has the longest YTA bar

Do these words tell you who was in the wrong?

Your answer:

## 4. Embeddings: search by meaning

An embedding turns each post into 384 numbers. Posts about similar things get similar numbers, so a search with a sentence also finds posts that use other words.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")  # the first run downloads the model, about 90 MB
embeddings = model.encode(posts["text"].tolist(), normalize_embeddings=True)
embeddings.shape  # Check: (785, 384)

In [ ]:
def search(sentence, n=5):
    """The n posts closest to a sentence."""
    query = model.encode([sentence], normalize_embeddings=True)[0]
    found = posts.assign(similarity=embeddings @ query)
    return found.sort_values("similarity", ascending=False).head(n)[["similarity", "verdict", "title"]]

search("My roommate keeps eating my food.")  # Check: the first post is about non halal foods

In [ ]:
# Example: how many posts use the word "roommate"? Which of the five posts above would a search for that word miss?
posts["text"].str.contains("roommate", case=False).sum()  # 24

In [ ]:
# Your turn: search with your own sentence about a topic from your project

In [ ]:
# Your turn: count the posts that contain the main word of your sentence
# Check: the search found posts without this word

Do the posts the search finds share a verdict?

Your answer:

## If you have time

### Precision and recall

For the label you are looking for, here yes:

- **Precision**: of the posts labeled yes, the share whose hand label is also yes.
- **Recall**: of the posts whose hand label is yes, the share labeled yes.

Compute both for `gemini`.

In [ ]:
# Example: precision of gemini: of the posts it labeled yes, the share whose hand label is also yes
tp = ((labels["gemini"] == "yes") & (labels["hand"] == "yes")).sum()
tp / (labels["gemini"] == "yes").sum()  # 0.43

In [ ]:
# Your turn: recall of gemini: of the posts whose hand label is yes, the share gemini labeled yes. Check: 0.75